In [2]:
import json
import re
import os
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import sqlglot
from sqlglot.expressions import Column, Table

# 1. 경로 설정 (notebooks 폴더에서 실행한다고 가정)
DEV_JSON_PATH = "/home/hyeonjin/thesis_refactored/data/raw/BIRD_dev/dev.json"
TABLES_JSON_PATH = "/home/hyeonjin/thesis_refactored/data/raw/BIRD_dev/dev_tables.json"
SCORE_JSON_PATH = "/home/hyeonjin/thesis_refactored/outputs/baselines/preliminary_vector_only/score_analysis_preliminary_vector_only.json"

/home/hyeonjin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def normalize_name(name):
    """특수문자와 공백을 제거한 순수 소문자 반환"""
    if not name: return ""
    return re.sub(r'[^a-z0-9]', '', str(name).lower())

def robust_extract(sql_query):
    """정답 SQL에서 Table과 Column을 추출하는 무결점 로직"""
    cols, tbls = set(), set()
    try:
        parsed = sqlglot.parse_one(sql_query, read="sqlite")
        cols = {node.name for node in parsed.find_all(Column)}
        tbls = {node.name for node in parsed.find_all(Table)}
    except Exception:
        pass
    if not cols:
        tokens = re.findall(r'`([^`]+)`|"([^"]+)"', sql_query)
        for t1, t2 in tokens:
            val = t1 or t2
            if val: cols.add(val)
    return {normalize_name(c) for c in cols}, {normalize_name(t) for t in tbls}

def regenerate_scores():
    print("🚀 1. 데이터 및 임베딩 모델 로드 중...")
    with open(DEV_JSON_PATH, 'r', encoding='utf-8') as f:
        dev_data = json.load(f)
    with open(TABLES_JSON_PATH, 'r', encoding='utf-8') as f:
        schema_map = {item['db_id']: item for item in json.load(f)}
    
    # 파이프라인 의존성 없이 로컬에서 모델 직접 로드 (2초 소요)
    encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    encoder.eval()

    score_analysis_data = []

    print("⚡ 2. Vector-Only 스코어 재계산 및 정답 라벨링 진행 중...")
    for idx, item in tqdm(enumerate(dev_data), total=len(dev_data)):
        q_id = item.get('question_id', idx)
        db_id = item['db_id']
        question = item['question']
        gold_sql = item['SQL']

        # 정답 추출
        gold_cols, gold_tbls = robust_extract(gold_sql)

        schema = schema_map.get(db_id, {})
        table_names = schema.get('table_names_original', [])
        column_names = schema.get('column_names_original', [])

        node_names = []
        node_texts = []

        # Table 노드 생성
        for t_name in table_names:
            node_names.append(t_name)
            node_texts.append(f"Table: {t_name}")

        # Column 노드 생성
        for t_idx, c_name in column_names:
            if t_idx < 0: continue
            t_name = table_names[t_idx]
            node_names.append(f"{t_name}.{c_name}")
            node_texts.append(f"Column: {c_name} in table {t_name}")

        # 임베딩 및 유사도 연산 (Vector Only 베이스라인 완벽 재현)
        with torch.no_grad():
            q_emb = encoder.encode([question], convert_to_tensor=True)
            n_embs = encoder.encode(node_texts, convert_to_tensor=True)
            scores = torch.nn.functional.cosine_similarity(q_emb, n_embs).cpu().tolist()

        # 스코어 저장 및 라벨링
        for name, score in zip(node_names, scores):
            is_gold = False
            if '.' in name:
                tbl, col = name.split('.', 1)
                if normalize_name(col) in gold_cols:
                    is_gold = True
            else:
                if normalize_name(name) in gold_tbls:
                    is_gold = True

            score_analysis_data.append({
                "query_id": q_id,
                "node_name": name,
                "score": score,
                "is_gold": is_gold
            })

    # 3. 정상적인 데이터로 파일 덮어쓰기
    os.makedirs(os.path.dirname(SCORE_JSON_PATH), exist_ok=True)
    with open(SCORE_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(score_analysis_data, f, indent=4, ensure_ascii=False)

    total_golds = sum(1 for row in score_analysis_data if row['is_gold'])
    print(f"✅ 재생성 완료! 정상적으로 수집된 총 노드: {len(score_analysis_data):,}개")
    print(f"✅ 그 중 Gold Node 갯수: {total_golds:,}개")

In [4]:
regenerate_scores()

🚀 1. 데이터 및 임베딩 모델 로드 중...
⚡ 2. Vector-Only 스코어 재계산 및 정답 라벨링 진행 중...


100%|██████████| 1534/1534 [01:33<00:00, 16.42it/s]


✅ 재생성 완료! 정상적으로 수집된 총 노드: 126,760개
✅ 그 중 Gold Node 갯수: 16,038개
